# <span style="color: #2BF507; background-color: #024E4C; padding: 15px">Roteiro 5 - Controle de Sistemas: PID</span> 

## <span id="Inicio">Conteúdo</span>
1. <a href="#Introducao">Controlador PID - <i>Proportional, Integrative, Derivative</i></a>  
  1.1 <a href="#Origem">Origem do Controlador</a>  
  1.2 <a href="#PID">Controlador PID</a>  
1. <a href="#PID_motor_dc">Controlador PID para Motor DC</a>  
  2.1 <a href="#Acoes">Ações das Componentes de Controle</a>  
  2.2 <a href="#Componentes">Componentes de um Sistema PID</a>  
  2.3 <a href="#Algoritmo">Algoritmo de Controle PID</a>  
  2.4 <a href="#Simulacao">Simulação do Controlador PID</a>  
1. <a href="#widgets">Widgets (componentes)</a>  
  3.1 <a href="#slider">Widget Slider-trackbar</a>

### <span style="color: #0040ff;" id="Introducao">1. Introdução: Controlador PID - <i>Proportional, Integrative, Derivative</i></span>

<img src="rot5_fig1.png" alt="Sistema de Controle PID" width="600" style="display: block; margin: 0 auto;">  

Esse tipo de controlador avalia o erro (diferença entre saída e *setpoint*)  baseado em: quão longe ele está do alvo (*setpoint*), quão rápido ele está variando e quão grande/pequeno ele está.

**Controle de Sistemas** é a espinha dorsal da automação moderna.
<div style="display: flex; justify-content: center; align-items: center; height: 400px;">
  <img src="rot5_fig2a.png" alt="Controle PID" width="450" >
  <img src="rot5_fig2b.png" alt="Controle PID" width="450" >  
</div>

<a href="#Inicio"><span style="color: #0040ff;">Voltar ao início</span></a>
#### <span style="color: #0040ff;" id="Origem">1.1 Origem do Controlador PID</span>

<img src="rot5_fig3.png" alt="Foto do busto do Nicolas Minorsky" style="float:right; width:150px;">  

Nicolas Minorsky (1885-1970), nascido na Rússia, publicou um artigo no _Journal of American Society of Naval Engineers_, vol. 42, No. 2, pp. 280-309, 1922, intitulado "*Directional Stability of Automatically Steered Bodies*" (Estabilidade Direcional de Corpos Dirigidos Automaticamente), no qual ele tratou da intuição do timoreiro ao conduzir uma embarcação, usando a posição indicada pela bússola, e da possibilidade de tornar automática a manobra da embarcação. Nesse artigo ele propõe um "controlador de três termos" e descarta a intuição do timoreiro e o uso somente da posição indicada pela bússola.  Ele afirma no artigo que além da **posição** indicada pela bússola o timoreiro também considera a **velocidade** da embarcação, além da percecpção de distúrbios diversos e imprevisíveis (ventos laterais, ondas etc.).  

<img src="rot5_fig4.png" alt="Erro da direção da embarcação: direção desejada - direção atual" style="float:left; width:300px;">

<p style="margin-top: 40px;"></p>

Equação diferencial para estimar o <b>erro de posição</b> da embarcação ($\alpha$), proposta por Minorsky: 

$$ A.\frac{d^2\alpha}{dt^2} + B.\frac{d\alpha}{dt} + k.\rho = D \tag {1} $$
onde:
* $A, B, k$ - são constantes  
* $\alpha$ - é o erro de posição (variável de saída, que nesse caso deve ser a menor possível)
* $D$ - é o distúrbio (ventos laterais)
* $\rho$ - é o ângulo do leme (variável de entrada que é manipulada pelo "controlador" -- timoreiro)

Este é um **modelo linear** válido para pequenos valores de erro de posição, $\alpha$.

Minorsky argumenta que além do ângulo do leme ($\rho$) também é necessário que o timoreiro ajuste o leme com certa velocidade angular ($d\rho/dt$), e assim ele usou também outra equação para descrever essa variável:
$$\frac{d\rho}{dt} = m.\alpha + n.\frac{d\alpha}{dt} + p.\frac{d^2\alpha}{dt^2} \tag {2}$$

Integrando ambos lados da equação em relação ao tempo, temos que o **ângulo do leme** é dado por:
$$\rho = \int (m.\alpha + n.\frac{d\alpha}{dt} + p.\frac{d^2\alpha}{dt^2}) dt \tag{3}$$

Substituindo (3) em (1): 
$$A.\frac{d^2\alpha}{dt^2} + B.\frac{d\alpha}{dt} + k.\int (m.\alpha + n.\frac{d\alpha}{dt} + p.\frac{d^2\alpha}{dt^2}) dt = D$$
$$A.\frac{d^2\alpha}{dt^2} + B.\frac{d\alpha}{dt} + \color{blue}{k.m\int \alpha \; dt + k.n.\alpha + k.p\frac{d\alpha}{dt}}\color{grey} = D \tag{4}$$
onde percebemos claramente as componentes do controle PID:
* **Proporcional**: $k.n.\alpha$
* **Integrativo**: $\;\;k.m\int \alpha \; dt$ (parcela da ação de controle que **garante erro estacionário nulo**, demonstrado pelo teorema do valor final aplicado à malha)
* **Derivativo**: $\;\;\;k.p\;{d\alpha}/{dt}$

<a href="#Inicio"><span style="color: #0040ff;">Voltar ao início</span></a>
#### <span style="color: #0040ff;" id="PID">1.2 Controlador PID</span>
Um controlador PID é um algoritmo de automação de processos, usado em máquinas e aparelhos para manter um sistema exatamente onde se deseja. Significando Proporcional, Integral e Derivativo, ele funciona como um ser humano dirigindo um carro e ajustando o pedal do acelerador para manter uma velocidade constante enquanto sobe e desce ladeiras.

1. **Proporcional** (o "Reagir", o "Agora")  
- *Ação*: analisa a que distância está a velocidade do automóvel de sua referência (100 km/h) e faz uma correção com base nessa diferença.  
- *Exemplo*: a velocidade de cruzeiro do carro é ajustada para 100 km/h. Se a velocidade cair para 70 km/h, a diferença é grande, então a parte **Proporcional** "pisa forte" no acelerador. À medida que a velocidade se aproxima dos 100 km/h, a pressão no pedal do acelerador diminui.
- *Desvantagem*: por si só, o controle proporcional geralmente deixa o carro um pouco abaixo, ou acima, da velocidade alvo. 

2. **Integral** (o "Histórico")
- *Ação*: analisa os erros ao longo do tempo. Se a parte Proporcional deixar o sistema ligeiramente fora do alvo, a parte Integral empurra suavemente o sistema para zerar completamente a diferença.
- *Exemplo*: embora o automóvel esteja quase a 100 km/h, ele está andando um pouco devagar demais por alguns segundos. A parte **Integral** diz: "Estamos abaixo da velocidade de cruzeiro há muito tempo, vamos adicionar um pouco mais de combustível para atingir o alvo exato." 

3. **Derivada** (a "Previsão", o "Futuro")
- *Ação*: antecipa o rumo que o sistema está tomando, observando a rapidez com que o valor está mudando (a taxa de mudança da velocidade do carro). Ele atua como um freio para evitar ultrapassar o alvo.
- *Exemplo*: ao perceber que a velocidade do carro está aumentando rapidamente em direção a 100 km/h, a parte **Derivativa** prevê que o carro ultrapassará os 100 km/h, então ela alivia o acelerador antes do carro realmente atingir o alvo, fazendo com que o carro permaneça na velocidade programada para o cruzeiro (referência).

##### <span style="color: #0040ff;">1.2.1 Diagrama de Blocos</span>  
<img src="rot5_fig5.png" alt="Diagrama de Blocos do Controlador PID e da planta" style="display:block; margin-left:auto; margin-right:auto; width:600px;">  

Sinal de controle da planta (ação de controle), $m(t)$, é uma soma de três termos. Usando o padrão ISA (*International Society of Automation* -- associação internacional, sem fins lucrativos, que desenvolve normas e padrões globais de segurança e automação industrial), temos: 
$$m(t) = K_P \left (e(t) + \frac{1}{T_I}\int_0^t e(\tau) d\tau + T_D \frac{de(t)}{dt} \right ) \tag{5}$$
onde:
* $K_P$ é o ganho proporcional;
* $K_P/T_I$ é o ganho integrativo e $T_I$ é o tempo integrativo;
* $K_P.T_D$ é o ganho derivativo e $T_D$ é o tempo derivativo.

Obs.: Os parâmetros $T_I$ e $T_D$ tem dimensões de tempo e podem naturalmente ser relacionados às constantes de tempo do controlador.  

Na malha com realimentação unitária negativa, o sinal **Erro**, $e(t)$, usado pelo controlador é: 
$$e(t) = r(t) - y(t)$$

##### <span style="color: #0040ff;">1.2.2 Função de Transferência do Controlador PID</span>
Representação idealizada (uma abstração útil ao entendimento do controlador PID, mas várias modificações devem ser feitas para se obter um controlador útil na prática):
<img src="rot5_fig6.png" alt="Diagrama de Blocos somente do Controlador PID" style="display:block; margin-left:auto; margin-right:auto; width:600px;">  
Considerando o sistema (planta) com condições iniciais nulas, vamos aplicar a Transformada de Laplace na equação (5):  

$$M(s) = K_P \left ( E(s) + \frac{1}{T_I}.\frac{E(s)}{s} + T_D .E(s) s \right )$$
$$\frac{M(s)}{E(s)} = K_P \left ( 1 + \frac{1}{T_I .s} + T_D . s \right ) \tag{6}$$

##### <span style="color: #0040ff;">1.2.3 Controle Proporcional</span>
A figura seguinte mostra as respostas da saída do sistema (processo ou planta), $G(s) = 1/(s + 1)^3$, a um degrau unitário usado na entrada  (sinal de referência) para um sistema com **controle proporcional** puro em diferentes configurações de ganho $(K_P=1, 2, 5)$.  

Devido a ausência de um termo no numerador do caminho direto (*feedforward*) $G(s)$, a saída nunca atingirá a referência (entrada) e, portanto, ficamos com valores diferentes de zero para o erro em **estado estacionário**. Fazendo a função de transferência do processo realimentado ser $G_{r}(s)$, e a realimentação (*feedback*) proporcional, temos $C(s) = K_P$ e a função de transferência da referência para o erro (controlador + planta) é:  $$G_{r}(s) = \frac{1}{1+C(s).G(s)} = \frac{1}{1+K_P.G(s)}$$  

<img src="rot5_fig7.png" alt="Diagrama de Blocos somente do Controlador PID" style="display:block; margin-left:auto; margin-right:auto; width:500px;"> 

Assumindo que a malha fechada seja estável, o **erro** $e(t)$ em regime permanente (estado estacionário) para uma referência (entrada) degrau unitário pode ser calculado pelo Teorema do Valor Final: $$\lim_{t \to \infty} g_{er}(t) = \lim_{s \to 0} s.G_{er}(s)$$
$$G_{r}(0) = \frac{1}{1+C(0)G(0)} = \frac{1}{1+K_P.G(0)}$$

Para o sistema da figura anterior, com ganhos $K_P = 1, 2$ e  $5$, o erro em estado estacionário correspondente será de $0,5; 0,33$ e $0,17$. O erro diminui com o aumento do ganho, mas o sistema também se torna mais oscilatório. O sistema fica instável para $K_P \ge 8$. Observe na figura anterior que o valor inicial do sinal de controle é igual ao ganho do controlador.

A última da parcela da equação (6) $T_D .s$ é não causal, de modo que em termos práticos não é implementável. Usamos uma aproximação para o termo de zero único para que ele tenha um polo além do zero:
$$T_D .s \approx \frac{T_D .s}{T_D .s+1}$$

Gerando a seguinte **Função de Transferência**: $\frac{M(s)}{E(s)} = K_P \left ( \frac{T_I .T_D .s^2 + T_I . s + 1}{T_I .s} \right ) =  K_P \left ( T_D .s + 1 +\frac{1}{T_I .s} \right ) $

<a href="#Inicio"><span style="color: #0040ff;">Voltar ao início</span></a>
### <span style="color: #0040ff;" id="PID_motor_dc">2. Controlador PID para Motor DC</span>

Um controlador **PID** (Proporcional, Integral e Derivativo) para motores DC é um sistema de malha fechada que ajusta a potência aplicada ao motor para atingir e manter com precisão uma velocidade, posição ou ângulo desejados. O controle minimiza o erro de estado permanente lendo sensores e enviando sinais precisos de correção à entrada (realimentação).  

#### <span style="color: #0040ff;" id="Acoes">2.1 Ações das Componentes de Controle </span> 
* **Ação Proporcional** (P): Corrige/reage o/ao erro atual de forma instantânea. Quanto maior for o $K_p$ maior será a capacidade de resposta (rapidez), mas também pode causar oscilação.  
* **Ação Integral** (I): Elimina o erro acumulado ao longo do tempo (erro de estado estacionário), garantindo que o motor chegue exatamente ao valor ou *setpoint* (posição ou velocidade) desejado.  
* **Ação Derivativa** (D): Antecipa o comportamento do motor com base na taxa de variação (derivada) do erro. Isso reduz oscilações bruscas e estabiliza o sistema mais rapidamente. Atua como um "freio" que amortece as oscilações e diminui o *overshoot*, tornando a resposta do motor mais suave.

#### <span style="color: #0040ff;" id="Componentes">2.2 Componentes de um Sistema PID </span> 
Para montar um controle prático precisaremos de pelo menos quatro elementos:  
* `Microcontrolador`: O cérebro do sistema (Arduino, ESP32, PIC, STM32 etc.) onde o algoritmo **PID** será processado.  
* `Driver de Potência` (ponte H, acionador de potência): Componente que recebe o sinal do microcontrolador e fornece a corrente necessária para o motor (ex: 2L293D, L298N, L9110, A4988, BTS7960, TB6612FNG etc.).  
* `Sensor` (*feedback*): Geralmente um encoder óptico ou magnético, que mede a velocidade atual (rpm) ou a posição do eixo do motor.  
* `Motor DC`: O atuador do sistema.  

Numa simulação de um controle preciso de **velocidade** ou da **posição**, geralmente implementamos um laço **PID** (Proporcional-Integral-Derivativo) em Python. Este algoritmo ajusta continuamente o ciclo de trabalho PWM com base na realimentação (*feedback*) de um codificador para atingir uma velocidade angular (RPM) alvo.  

O modelo de motor utilizado nesta simulação simplifica significativamente o comportamento de um **Motor DC** real. Neste modelo, assumimos uma relação linear direta entre o sinal de controle (tensão de armadura) e a velocidade angular do motor, ignorando diversas complexidades presentes nos sistemas reais. Motores reais possuem atrito, inércia e respostas não lineares que não são contabilizados nesta simulação.  

#### <span style="color: #0040ff;" id="Algoritmo">2.3 Algoritmo de Controle PID</span>
O cálculo do sinal de controle (variável manipulada, que geralmente pode ser um sinal PWM para o *driver*) é dado por:  

$ v_a(t) = K_p . e(t) + K_i . \int_0^t e(\tau) d \tau + K_d . \frac{d e(t)}{dt} $  

onde:
* $e(t)$ é o erro atual (diferença entre o valor desejado e o medido).  
* $K_p$, $K_i$ e $K_d$ são os ganhos de sintonia que devem ser ajustados para o seu sistema realimentado específico: motor com *encoder* para medir rpm's na saída.  

$v_a(t) = K_p . e(t) + K_i . \int_0^t e(\tau) d \tau + K_d . \frac {d e(t)} {dt} $

#### <span style="color: #0040ff;" id="Simulacao">2.4 Simulação do Controlador PID</span>
Vamos usar elementos interativos do Python, *widgets*. Eles permitem a construção de interfaces amigáveis para usuários.  

Usaremos o *slider* ou barra de rastreamento para variar linearmente diversos parâmetros da simulação: ganhos, *setpoint*, duração etc.

##### Slider/trackbar 
Usado para entrada de valores flutuantes em uma faixa específica.  
`FloatSlider(value=0.1, min=0, max=25.0, step=0.05, description='Kp (Proporcional)')`  

Parâmetros:  

   value : float
*    posição do *slider*

   min : float
*    menor valor da faixa do *slider*

   max : float
*    maior valor da faixa do *slider*

   step : float
*    passo de incremento/decremento do valor na faixa do *slider*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, FloatSlider
import ipywidgets as widgets

# Parâmetros físicos do motor DC
J = 0.05  # Inércia do rotor
b = 0.1   # Atrito viscoso
K = 0.05  # Constante de torque/força contra-eletromotriz
R = 1.0   # Resistência do indutor
L = 0.5   # Indutância do motor

def simular_motor_pid(Kp, Ki, Kd, setpoint, tempo_simulacao):
    dt = 0.01  # Passo de tempo
    n_pontos = int(tempo_simulacao / dt)
    
    tempo = np.linspace(0, tempo_simulacao, n_pontos)
    velocidade = np.zeros(n_pontos)
    erro = np.zeros(n_pontos)
    saida_pid = np.zeros(n_pontos)
    
    # Variáveis do PID
    integral = 0
    erro_anterior = 0
    
    # Simulação passo a passo (Euler)
    for i in range(1, n_pontos):
        erro[i] = setpoint - velocidade[i-1]
        integral += erro[i] * dt
        derivada = (erro[i] - erro_anterior) / dt
        
        # Ação de controle: Saída do controlador PID (tensão de armadura aplicada)
        saida_pid[i] = Kp * erro[i] + Ki * integral + Kd * derivada
        
        # Dinâmica do motor: dv/dt = (-b/J)*v + (K/J)*V
        d_velocidade = (-b/J) * velocidade[i-1] + (K/J) * saida_pid[i]
        velocidade[i] = velocidade[i-1] + d_velocidade * dt
        erro_anterior = erro[i]

    # Gráfico: setpoint e variável controlada (velocidade) em função do tempo
    plt.figure(figsize=(10, 5))
    plt.plot(tempo, velocidade, label='Velocidade Real', color='blue', linewidth=2)
    plt.axhline(y=setpoint, color='red', linestyle='--', label='Setpoint')
    plt.title('Simulação de Controle PID de um Motor DC')
    plt.xlabel('Tempo (s)'); plt.ylabel('Velocidade (m/s)')
    plt.ylim(0, setpoint * 1.2 if setpoint > 0 else 1.2)
    plt.grid(True); plt.legend(loc='lower right'); plt.show()

# Widgets Interativos
controle_interativo = interact(
    simular_motor_pid,
    Kp = FloatSlider(value=0.1, min=0, max=25.0, step=0.05, description='Kp (Proporcional)'),
    Ki = FloatSlider(value=0.01, min=0, max=1.0, step=0.01, description='Ki (Integral)'),
    Kd = FloatSlider(value=0.0, min=0, max=1.0, step=0.01, description='Kd (Derivativo)'),
    setpoint = FloatSlider(value=10.0, min=5.0, max=30.0, step=1.0, description='Veloc. Alvo'),
    tempo_simulacao = FloatSlider(value=2.0, min=1.0, max=5.0, step=0.25, description='Tempo (s)')
)

interactive(children=(FloatSlider(value=0.1, description='Kp (Proporcional)', max=25.0, step=0.05), FloatSlide…

<a href="#Inicio"><span style="color: #0040ff;">Voltar ao início</span></a>
### <span style="color: #0040ff;" id="widgets">3. Widgets (componentes)</span>

Quando se está plotando gráficos em python, às vezes é interessante alterar parâmetros na equação para se observar como ela afeta a saída do programa.

Porém não é conveniente voltar ao código, alterar os parâmetros, executar o código e aguardar a resposta, principalmente em códigos em que o tempo de resposta pode demorar alguns segundos. O pacote ```ipywidgets``` pode ser uma alternativa que permite uma maior interatividade com o usuário e praticidade.

O pacote ```ipywidgets ``` é uma biblioteca Python desenvolvida para criar interfaces gráficas interativas em documentos do *Jupyter Notebook* ou do *JupyterLab*. O principal objetivo desse pacote é permitir que o usuário interaja com programas e visualizações de maneira dinâmica, sem a necessidade de desenvolver interfaces complexas utilizando ferramentas externas.

A biblioteca fornece diversos componentes gráficos, chamados de **widgets**, que possibilitam modificar parâmetros de programas enquanto eles estão em execução, tornando a análise de dados, simulações matemáticas e experimentos computacionais muito mais intuitivos.

#### <span style="color: #0040ff;" id="slider">3.1 Widget Slider-trackbar</span>

Os *sliders* (deslizantes) são uma das formas mais intuitivas de se modificar parâmetros a partir de interfaes gráficas, pois permitem que se altere valores de forma bem controlada e dentro de uma faixa definida (mínimo e máximo) com passos também alteráveis.

Aqui está um exemplo:

In [20]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def atualiza(ampl_pico, freq, fase):            # atualizar é a função que permite que os valores sejam constantemente modificados.
    x = np.linspace(0, 10, 1000)
    y = ampl_pico*np.sin(2*np.pi*freq*x - fase) # definindo a senoide

    plt.figure(figsize=(6, 3))                  # mostrando a senoide
    plt.plot(x, y)
    plt.title("Senoide com ajuste em tempo real")
    plt.xlabel("x"); plt.ylabel("y"); plt.grid(True); plt.ylim(-10, 10)

# Criação dos sliders
slider_amplitude = widgets.FloatSlider(         # Criando um slider para valores reais
    value=1,                                    # Valor inicial
    min=0,                                      # Valor mínimo
    max=10,                                     # Valor máximo
    step=0.1,                                   # Tamanho dos intervalos
    description='Amplitude'                     # Nome quse dará ao slider
)

slider_frequencia = widgets.FloatSlider(
    value=5,
    min=0.1,
    max=10,
    step=0.1,
    description='Freq. (Hz)'
)

slider_fase = widgets.FloatSlider(
    value=0,
    min=0,
    max=2*np.pi,
    step=0.1,
    description='Fase (rad)'
)

# Liga os sliders à função
widgets.interactive(                    # Função que relaciona as variáveis do valor do slider 
    atualiza,                           # com as variáveis da equação
    ampl_pico=slider_amplitude,
    freq=slider_frequencia,
    fase=slider_fase
)

interactive(children=(FloatSlider(value=1.0, description='Amplitude', max=10.0), FloatSlider(value=5.0, descri…

Mas o *slider* também pode ser exibido na vertical, mudando o parâmetro de orientação, como mostrado a seguir. 

In [ ]:
widgets.FloatSlider(value=5, 
                    min=0, 
                    max=10, 
                    orientation='vertical', 
                    layout=widgets.Layout(width='50px', height='150px'))

O ```IntSlider``` representa *sliders* para valores inteiros, o ```FloatLogSlider``` representa valores em escala logaritímica e ```RangeSlider``` tanto para valores inteiros e reais que representam uma faixa de valores

### Exemplos:

In [ ]:
#Slider para valores inteiros

widgets.IntSlider(
    value=5,
    min=0,
    max=10,
    step=1,
    description='Teste:',
    disabled=False,             #Permite o slider ser disabilitado e abilitado
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='02d'
)

In [ ]:
#Slider para valores logarítimos

widgets.FloatLogSlider(
    value=10,
    base=10,
    min=-10,   # expoente mínimo da base
    max=10,    # expoente máximo da base
    step=0.1,  # passo do expoente
    description='Escala Log'
)

In [ ]:
#Slider para faixa de valores inteiros

widgets.IntRangeSlider(
    value=[3, 7],
    min=0,
    max=10,
    step=1,
    description='Range Int:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
)

In [ ]:
#Slider para faixa de valores reais

widgets.FloatRangeSlider(
    value=[5, 7.5],
    min=0,
    max=10.0,
    step=0.1,
    description='Range Float:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

<a href="#Inicio"><span style="color: #0040ff;">Voltar ao início</span></a>
### Fonte: 
1. Ipywidgets; "Jupyter Widgets 8.1.8 documentation"; 2023; disponível em: https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html; acesso em: 25/05/2026.


### Exercício Final